# AMPS I · Week 06 — Lab
## Boundary conditions: constants, lifting, Neumann and Robin terms, nullspaces, multi-material meshes

| | |
|---|---|
| **Lecture sections** | Part A (mean estimate, trace, partial Dirichlet) · Part B (lifting, DOF location) · Part C (`ds`, nullspace, subdomains, Gmsh) · Part D (LLM) |
| **Time** | ≈ 80 min in class |
| **You will** | compute the mean-estimate, trace and partial-Dirichlet constants · apply a Dirichlet condition by hand and compare with DOLFINx · verify a solver with Dirichlet, Neumann and Robin conditions by MMS · solve a pure Neumann problem with a PETSc nullspace · build a two-material mesh with Gmsh and see why the mesh must follow the interface |

**Reminder.** `%%fenicsx` cells run in a separate MPI process and are the only cells where `basix`, `ufl`, `dolfinx`
(and here also `scipy`, `petsc4py`, `gmsh`) are used. Ordinary cells have NumPy and matplotlib only. Data passes through `/tmp/fnx/`.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## 1 · Constants that make Poisson coercive  *(lecture §A3–A7)*
P1 on the unit square, dense matrices on one process. With $K$ (stiffness), $M$ (mass), $B_{ij}=\int_{\partial\Omega}\phi_j\phi_i$ and $G=K+M$:

- mean estimate $\|v-\bar v\|_{L^2}\le C|v|_{H^1}$: best $C_h=1/\sqrt{\lambda_2(K,M)}$ (the first eigenvalue is $0$, the constants)
- trace inequality $\|v\|_{L^2(\partial\Omega)}\le C_T\|v\|_{H^1}$: best $C_{T,h}^2=\lambda_{\max}(B,G)$
- coercivity of Poisson on $\{v=0$ on $\Gamma_D\}$: $\gamma_h=\lambda_{\min}$ of $(K,G)$ with the rows and columns of $\Gamma_D$ removed

In [ ]:
%%fenicsx -np 1
from mpi4py import MPI
import numpy as np
import scipy.linalg
import ufl
from dolfinx import fem, mesh

def eig(A, B, i):
    return scipy.linalg.eigh(A, B, eigvals_only=True, subset_by_index=[i, i])[0]

def sub(A, rm):
    keep = np.setdiff1d(np.arange(len(A)), rm)
    return A[np.ix_(keep, keep)]

for N in (4, 8, 16, 32):
    msh = mesh.create_unit_square(MPI.COMM_WORLD, N, N)
    V = fem.functionspace(msh, ("Lagrange", 1))
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    dense = lambda form: fem.assemble_matrix(fem.form(form)).to_dense()
    K, M, B = dense(ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx), dense(u * v * ufl.dx), dense(u * v * ufl.ds)
    G = K + M
    X = V.tabulate_dof_coordinates()
    all_bnd = np.where(np.isclose(X[:, 0], 0) | np.isclose(X[:, 0], 1) | np.isclose(X[:, 1], 0) | np.isclose(X[:, 1], 1))[0]
    left = np.where(np.isclose(X[:, 0], 0))[0]
    print(f"N={N:2d}: C_mean = {1 / np.sqrt(eig(K, M, 1)):.4f}   C_T^2 = {eig(B, G, len(G) - 1):.4f}   "
          f"gamma_h: Gamma_D = whole boundary {eig(sub(K, all_bnd), sub(G, all_bnd), 0):.4f},  "
          f"left side {eig(sub(K, left), sub(G, left), 0):.4f}")
print(f"limits: 1/pi = {1 / np.pi:.4f},  2pi^2/(1+2pi^2) = {2 * np.pi**2 / (1 + 2 * np.pi**2):.4f},  "
      f"(pi^2/4)/(1+pi^2/4) = {np.pi**2 / 4 / (1 + np.pi**2 / 4):.4f}")

> **✏️ Your turn (5 min).** Let $\Gamma_D=\{x=0,\ 0\le y\le\ell\}$. Compute $\gamma_h$ for $\ell=1,\tfrac12,\tfrac14,\tfrac18,\tfrac1{16}$ with $N=16$ and $N=32$.
> Does $\gamma_h$ depend on $\ell$? On $h$? Which factor in the proof of the partial-Dirichlet theorem (lecture §A7) explains the trend?

In [ ]:
%%fenicsx -np 1
# TODO: gamma_h for Dirichlet segments of length l on x = 0

## 2 · A Dirichlet condition by hand  *(lecture §B1–B3)*
$-\Delta u=1$ on a $4\times4$ mesh, $u=g=1+y$ on the left side, $\partial_nu=0$ elsewhere. Split the DOFs into free ($I$) and Dirichlet ($D$):
$A_{II}\mathbf u_I=\mathbf b_I-A_{ID}\mathbf g$. Compare with the DOLFINx pipeline `assemble_matrix(a, bcs)` → `apply_lifting` → `set_bc`.

In [ ]:
%%fenicsx -np 1
from mpi4py import MPI
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import assemble_matrix, assemble_vector, apply_lifting, set_bc

msh = mesh.create_unit_square(MPI.COMM_WORLD, 4, 4)
V = fem.functionspace(msh, ("Lagrange", 1))
u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
a = fem.form(ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx)
L = fem.form(1.0 * v * ufl.dx)
X = V.tabulate_dof_coordinates()

# locating the Dirichlet DOFs: three ways
msh.topology.create_connectivity(1, 2)
left_facets = mesh.locate_entities_boundary(msh, 1, lambda x: np.isclose(x[0], 0))
D = fem.locate_dofs_topological(V, 1, left_facets)
print("topological:", len(D), "  geometrical isclose:", len(fem.locate_dofs_geometrical(V, lambda x: np.isclose(x[0], 0))),
      "  geometrical x == 0:", len(fem.locate_dofs_geometrical(V, lambda x: x[0] == 0.0)),
      f"   (x-coordinates of D: {X[D, 0]})")

# by hand: full matrix, then eliminate
K = fem.assemble_matrix(a).to_dense()
F = fem.assemble_vector(L).array
I = np.setdiff1d(np.arange(len(K)), D)
g = 1 + X[D, 1]
u_hand = np.zeros(len(K))
u_hand[D] = g
u_hand[I] = np.linalg.solve(K[np.ix_(I, I)], F[I] - K[np.ix_(I, D)] @ g)

# DOLFINx
gD = fem.Function(V); gD.interpolate(lambda x: 1 + x[1])
bc = fem.dirichletbc(gD, D)
A = assemble_matrix(a, bcs=[bc]); A.assemble()
b = assemble_vector(L)
apply_lifting(b, [a], bcs=[[bc]])
b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
set_bc(b, [bc])
Ad = A.convert("dense").getDenseArray()
print("row of a Dirichlet DOF:", np.round(Ad[D[0]], 3)[np.abs(Ad[D[0]]) > 0], "  column zeroed:", np.allclose(Ad[I][:, D], 0),
      "  symmetric:", np.allclose(Ad, Ad.T))
print("b_D == g:", np.allclose(b.array[D], g), "   b_I == F_I - K_ID g:", np.allclose(b.array[I], F[I] - K[np.ix_(I, D)] @ g))
ksp = PETSc.KSP().create(msh.comm); ksp.setOperators(A); ksp.setType("preonly"); ksp.getPC().setType("lu")
uh = fem.Function(V); ksp.solve(b, uh.x.petsc_vec)
print("max |u_DOLFINx - u_hand| =", np.abs(uh.x.array - u_hand).max())

> **✏️ Your turn (5 min).** Repeat the DOLFINx part **without** `apply_lifting`. Which entries of `b` change, which DOFs of the solution are
> still correct, and what problem does the result actually solve?

In [ ]:
%%fenicsx -np 1
# TODO: the DOLFINx pipeline without apply_lifting

## 3 · MMS with Dirichlet, Neumann and Robin conditions  *(lecture §C1–C2)*
$-\Delta u=f$ with $u=g_D$ on $x=0$, $\partial_nu=g_N$ on $y=0,1$, $\partial_nu+\alpha u=r$ on $x=1$ ($\alpha=10$), and
$\tilde u=1+\sin(\pi x)e^y+xy^2$. All data come from $\tilde u$ by UFL; the normal from `ufl.FacetNormal`. Two processes.

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import assemble_matrix, assemble_vector, apply_lifting, set_bc

def norms(msh, e):
    """(L2 norm, H1 seminorm) of a UFL expression e, summed over all ranks."""
    s = lambda f: np.sqrt(msh.comm.allreduce(fem.assemble_scalar(fem.form(f * ufl.dx)), op=MPI.SUM))
    return s(ufl.inner(e, e)), s(ufl.inner(ufl.grad(e), ufl.grad(e)))

def table(Ns, E, label=""):
    E = np.asarray(E); r = np.log2(E[:-1] / E[1:])
    for i, N in enumerate(Ns):
        print(f"{label} N={N:3d}  L2 {E[i, 0]:.3e}  H1 {E[i, 1]:.3e}" + (f"   rates {r[i-1, 0]:.2f} {r[i-1, 1]:.2f}" if i else ""))

LEFT, RIGHT, BOTTOM, TOP = 1, 2, 3, 4

def side_tags(msh):
    """Facet tags 1-4 for x=0, x=1, y=0, y=1 (indices sorted, as meshtags expects)."""
    fdim = msh.topology.dim - 1
    msh.topology.create_connectivity(fdim, msh.topology.dim)
    facets, values = [], []
    for tag, marker in ((LEFT, lambda x: np.isclose(x[0], 0)), (RIGHT, lambda x: np.isclose(x[0], 1)),
                        (BOTTOM, lambda x: np.isclose(x[1], 0)), (TOP, lambda x: np.isclose(x[1], 1))):
        f = mesh.locate_entities_boundary(msh, fdim, marker)
        facets.append(f); values.append(np.full(len(f), tag, dtype=np.int32))
    facets, values = np.hstack(facets).astype(np.int32), np.hstack(values)
    order = np.argsort(facets)
    return mesh.meshtags(msh, fdim, facets[order], values[order])

def solve_lu(a, L, bcs, V):
    """assemble_matrix(bcs) → assemble_vector → apply_lifting → ghostUpdate → set_bc → LU."""
    a, L = fem.form(a), fem.form(L)
    A = assemble_matrix(a, bcs=bcs); A.assemble()
    b = assemble_vector(L)
    apply_lifting(b, [a], bcs=[bcs])
    b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
    set_bc(b, bcs)
    ksp = PETSc.KSP().create(V.mesh.comm)
    ksp.setOperators(A); ksp.setType("preonly"); ksp.getPC().setType("lu")
    uh = fem.Function(V)
    ksp.solve(b, uh.x.petsc_vec); uh.x.scatter_forward()
    reason = ksp.getConvergedReason()
    ksp.destroy(); A.destroy(); b.destroy()
    assert reason > 0, reason
    return uh

ALPHA = 10.0
Ns = [8, 16, 32, 64]
for k in (1, 2):
    E = []
    for N in Ns:
        msh = mesh.create_unit_square(MPI.COMM_WORLD, N, N)
        ft = side_tags(msh)
        ds = ufl.Measure("ds", domain=msh, subdomain_data=ft)
        if k == 1 and N == 8:
            lengths = [msh.comm.allreduce(fem.assemble_scalar(fem.form(1.0 * ds(i))), op=MPI.SUM) for i in (1, 2, 3, 4)]
            if msh.comm.rank == 0:
                print("lengths of the tagged sides:", np.round(lengths, 12))
        V = fem.functionspace(msh, ("Lagrange", k))
        x, n = ufl.SpatialCoordinate(msh), ufl.FacetNormal(msh)
        u_ex = 1 + ufl.sin(ufl.pi * x[0]) * ufl.exp(x[1]) + x[0] * x[1] ** 2
        f = -ufl.div(ufl.grad(u_ex))
        gN = ufl.dot(ufl.grad(u_ex), n)
        r = ufl.dot(ufl.grad(u_ex), n) + ALPHA * u_ex
        u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
        a = ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx + ALPHA * u * v * ds(RIGHT)
        L = f * v * ufl.dx + gN * v * ds(BOTTOM) + gN * v * ds(TOP) + r * v * ds(RIGHT)
        gD = fem.Function(V); gD.interpolate(fem.Expression(u_ex, V.element.interpolation_points))
        bc = fem.dirichletbc(gD, fem.locate_dofs_topological(V, 1, ft.find(LEFT)))
        uh = solve_lu(a, L, [bc], V)
        E.append(norms(msh, uh - u_ex))
    if MPI.COMM_WORLD.rank == 0:
        table(Ns, E, f"P{k}")

> **✏️ Your turn (5 min).** Replace the Dirichlet condition on $x=0$ by a second Robin condition ($\alpha=10$, data from $\tilde u$), so that
> there is **no** Dirichlet boundary at all. Is the problem still well posed (lecture §A9)? Do the rates change? P1 only.

In [ ]:
%%fenicsx -np 2
# TODO: Robin on x = 0 and x = 1, Neumann on y = 0, 1 — no Dirichlet condition

## 4 · Pure Neumann with a PETSc nullspace  *(lecture §C3–C4)*
$-\Delta u=f$, $\partial_nu=g$ on all of $\partial\Omega$, $\tilde u=\cos\pi x\cos\pi y+x^2y$. First check compatibility, then tell PETSc about the
constant nullspace, solve with CG + hypre, and remove the mean afterwards.

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import assemble_matrix, assemble_vector, apply_lifting, set_bc

def norms(msh, e):
    """(L2 norm, H1 seminorm) of a UFL expression e, summed over all ranks."""
    s = lambda f: np.sqrt(msh.comm.allreduce(fem.assemble_scalar(fem.form(f * ufl.dx)), op=MPI.SUM))
    return s(ufl.inner(e, e)), s(ufl.inner(ufl.grad(e), ufl.grad(e)))

def table(Ns, E, label=""):
    E = np.asarray(E); r = np.log2(E[:-1] / E[1:])
    for i, N in enumerate(Ns):
        print(f"{label} N={N:3d}  L2 {E[i, 0]:.3e}  H1 {E[i, 1]:.3e}" + (f"   rates {r[i-1, 0]:.2f} {r[i-1, 1]:.2f}" if i else ""))

comm = MPI.COMM_WORLD
Ns, E = [8, 16, 32, 64], []
for N in Ns:
    msh = mesh.create_unit_square(comm, N, N)
    V = fem.functionspace(msh, ("Lagrange", 1))
    x, n = ufl.SpatialCoordinate(msh), ufl.FacetNormal(msh)
    u_ex = ufl.cos(ufl.pi * x[0]) * ufl.cos(ufl.pi * x[1]) + x[0] ** 2 * x[1]
    f, g = -ufl.div(ufl.grad(u_ex)), ufl.dot(ufl.grad(u_ex), n)
    integral = lambda q: comm.allreduce(fem.assemble_scalar(fem.form(q)), op=MPI.SUM)
    compat = integral(f * ufl.dx + g * ufl.ds)
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    A = assemble_matrix(fem.form(ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx)); A.assemble()
    b = assemble_vector(fem.form(f * v * ufl.dx + g * v * ufl.ds))
    b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
    ns = PETSc.NullSpace().create(constant=True, comm=comm)
    A.setNullSpace(ns)
    ns.remove(b)
    ksp = PETSc.KSP().create(comm); ksp.setOperators(A)
    ksp.setType("cg"); ksp.getPC().setType("hypre"); ksp.setTolerances(rtol=1e-10)
    uh = fem.Function(V)
    ksp.solve(b, uh.x.petsc_vec); uh.x.scatter_forward()
    mean_before = integral(uh * ufl.dx)
    uh.x.array[:] -= mean_before                       # now int u_h = 0
    mean_after = integral(uh * ufl.dx)                 # collective: computed on every rank, printed on rank 0
    e = uh - (u_ex - integral(u_ex * ufl.dx))          # compare with the zero-mean exact solution
    E.append(norms(msh, e))
    if comm.rank == 0:
        print(f"N={N:2d}: int f + int g = {compat:+.1e}   CG its {ksp.getIterationNumber()}  reason {ksp.getConvergedReason()}"
              f"   int u_h before / after mean removal {mean_before:+.1e} / {mean_after:+.1e}")
    ksp.destroy(); A.destroy(); b.destroy()
if comm.rank == 0:
    table(Ns, E, "P1")

> **✏️ Your turn (5 min).** Make the data incompatible: $f\to f+1$ (so $\int f+\int g=1$). Solve (i) with the nullspace and `ns.remove(b)` as above,
> (ii) by **pinning** the DOF at the corner $(0,0)$ to $\tilde u(0,0)$ with a Dirichlet condition and LU. Compare the KSP reasons, the $L^2$ errors
> (after removing the mean) and $\max|u_h|$ for $N=8,\dots,64$. Which result would you notice without an exact solution?

In [ ]:
%%fenicsx -np 2
# TODO: incompatible data: nullspace + remove(b) vs one pinned DOF

## 5 · Two materials and Gmsh  *(lecture §C5–C8)*
Unit disk, $k=10$ for $r<\tfrac12$ (tag 1) and $k=1$ outside (tag 2), $u=0$ on $r=1$ (facet tag 10), exact solution $u=\cos(\pi r/2)/k+c$.
First one mesh on one process for a picture, then the convergence study on two processes.

In [ ]:
%%fenicsx -np 1
from mpi4py import MPI
import numpy as np
import ufl
import gmsh
from dolfinx import fem, mesh
from dolfinx.io import gmsh as gmshio
from dolfinx.fem.petsc import LinearProblem

def disk(h, interface=True, order=1):
    """Unit disk; with interface=True the circle r = 1/2 is part of the mesh (tags 1 = inclusion, 2 = matrix).
    Facet tag 10 = outer circle. The Gmsh model is built on rank 0 only."""
    gmsh.initialize(); gmsh.option.setNumber("General.Terminal", 0)
    if MPI.COMM_WORLD.rank == 0:
        gmsh.model.add("disk")
        outer = gmsh.model.occ.addDisk(0, 0, 0, 1, 1)
        if interface:
            inner = gmsh.model.occ.addDisk(0, 0, 0, 0.5, 0.5)
            gmsh.model.occ.fragment([(2, outer)], [(2, inner)])        # conforming interface
        gmsh.model.occ.synchronize()
        surf = sorted((gmsh.model.occ.getMass(2, s), s) for _, s in gmsh.model.getEntities(2))   # smallest first
        for tag, (_, s) in enumerate(surf, start=1):
            gmsh.model.addPhysicalGroup(2, [s], tag)
        bnd = gmsh.model.getBoundary(gmsh.model.getEntities(2), combined=True, oriented=False)
        gmsh.model.addPhysicalGroup(1, [c for _, c in bnd], 10)
        gmsh.option.setNumber("Mesh.CharacteristicLengthMax", h)
        gmsh.model.mesh.generate(2)
        if order == 2:
            gmsh.model.mesh.setOrder(2)                                # curved (isoparametric) triangles
    md = gmshio.model_to_mesh(gmsh.model, MPI.COMM_WORLD, 0, gdim=2)
    gmsh.finalize()
    return md.mesh, md.cell_tags, md.facet_tags

K_IN, K_OUT = 10.0, 1.0

def disk_problem(msh, ct, ft, k, interface=True):
    """-div(kappa grad u) = f in the unit disk, u = 0 on r = 1, kappa = 10 for r < 1/2 and 1 outside.
    Exact solution u = cos(pi r / 2) / kappa + c (continuous u and flux). Returns u_h and the exact UFL expression."""
    Q = fem.functionspace(msh, ("DG", 0)); kappa = fem.Function(Q)
    if interface:                                                      # from the Gmsh cell tags
        kappa.x.array[ct.find(1)] = K_IN; kappa.x.array[ct.find(2)] = K_OUT
    else:                                                              # mesh ignores the interface: use cell midpoints
        nc = msh.topology.index_map(2).size_local + msh.topology.index_map(2).num_ghosts
        mid = mesh.compute_midpoints(msh, 2, np.arange(nc, dtype=np.int32))
        kappa.x.array[:] = np.where(np.linalg.norm(mid[:, :2], axis=1) < 0.5, K_IN, K_OUT)
    V = fem.functionspace(msh, ("Lagrange", k))
    x = ufl.SpatialCoordinate(msh); r = ufl.sqrt(x[0] ** 2 + x[1] ** 2); phi = ufl.cos(ufl.pi * r / 2)
    u_ex = ufl.conditional(ufl.lt(r, 0.5), phi / K_IN + np.cos(np.pi / 4) * (1 / K_OUT - 1 / K_IN), phi / K_OUT)
    f = ufl.pi / 2 * ufl.sin(ufl.pi * r / 2) / r + (ufl.pi / 2) ** 2 * phi     # = -Δφ, the same in both materials
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    msh.topology.create_connectivity(1, 2)
    bc = fem.dirichletbc(0.0, fem.locate_dofs_topological(V, 1, ft.find(10)), V)
    uh = LinearProblem(kappa * ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx, f * v * ufl.dx, bcs=[bc],
                       petsc_options_prefix=f"disk_{k}_{int(interface)}_{msh.topology.index_map(2).size_global}_",
                       petsc_options={"ksp_type": "preonly", "pc_type": "lu"}).solve()
    return uh, u_ex

msh, ct, ft = disk(0.1)
print(f"Gmsh {gmsh.__version__}: {msh.topology.index_map(2).size_local} cells — inclusion {len(ct.find(1))}, "
      f"matrix {len(ct.find(2))}; outer facets {len(ft.find(10))}")
uh, u_ex = disk_problem(msh, ct, ft, 1)
tags = np.zeros(msh.topology.index_map(2).size_local, dtype=np.int32); tags[ct.indices] = ct.values
np.savez("/tmp/fnx/disk.npz", x=msh.geometry.x[:, :2], cells=msh.geometry.dofmaps[0], tags=tags, u=uh.x.array)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as mtri

d = np.load("/tmp/fnx/disk.npz")
tri = mtri.Triangulation(d["x"][:, 0], d["x"][:, 1], d["cells"])
fig, axs = plt.subplots(1, 3, figsize=(13, 4))
axs[0].tripcolor(tri, facecolors=d["tags"], cmap="Pastel1", edgecolors="0.4", linewidth=0.2)
axs[0].set_title("cell tags (1 = inclusion, 2 = matrix)")
tc = axs[1].tricontourf(tri, d["u"], 20); fig.colorbar(tc, ax=axs[1]); axs[1].set_title("$u_h$ (P1)")
r = np.hypot(d["x"][:, 0], d["x"][:, 1]); o = np.argsort(r)
axs[2].plot(r[o], d["u"][o], ".", ms=2); axs[2].axvline(0.5, color="k", lw=0.8, ls=":")
axs[2].set_xlabel("$r$"); axs[2].set_title("$u_h$ against $r$: kink at the interface")
for ax in axs[:2]:
    ax.set_aspect("equal"); ax.axis("off")
fig.tight_layout(); plt.show()

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import numpy as np
import ufl
import gmsh
from dolfinx import fem, mesh
from dolfinx.io import gmsh as gmshio
from dolfinx.fem.petsc import LinearProblem

def disk(h, interface=True, order=1):
    """Unit disk; with interface=True the circle r = 1/2 is part of the mesh (tags 1 = inclusion, 2 = matrix).
    Facet tag 10 = outer circle. The Gmsh model is built on rank 0 only."""
    gmsh.initialize(); gmsh.option.setNumber("General.Terminal", 0)
    if MPI.COMM_WORLD.rank == 0:
        gmsh.model.add("disk")
        outer = gmsh.model.occ.addDisk(0, 0, 0, 1, 1)
        if interface:
            inner = gmsh.model.occ.addDisk(0, 0, 0, 0.5, 0.5)
            gmsh.model.occ.fragment([(2, outer)], [(2, inner)])        # conforming interface
        gmsh.model.occ.synchronize()
        surf = sorted((gmsh.model.occ.getMass(2, s), s) for _, s in gmsh.model.getEntities(2))   # smallest first
        for tag, (_, s) in enumerate(surf, start=1):
            gmsh.model.addPhysicalGroup(2, [s], tag)
        bnd = gmsh.model.getBoundary(gmsh.model.getEntities(2), combined=True, oriented=False)
        gmsh.model.addPhysicalGroup(1, [c for _, c in bnd], 10)
        gmsh.option.setNumber("Mesh.CharacteristicLengthMax", h)
        gmsh.model.mesh.generate(2)
        if order == 2:
            gmsh.model.mesh.setOrder(2)                                # curved (isoparametric) triangles
    md = gmshio.model_to_mesh(gmsh.model, MPI.COMM_WORLD, 0, gdim=2)
    gmsh.finalize()
    return md.mesh, md.cell_tags, md.facet_tags

K_IN, K_OUT = 10.0, 1.0

def disk_problem(msh, ct, ft, k, interface=True):
    """-div(kappa grad u) = f in the unit disk, u = 0 on r = 1, kappa = 10 for r < 1/2 and 1 outside.
    Exact solution u = cos(pi r / 2) / kappa + c (continuous u and flux). Returns u_h and the exact UFL expression."""
    Q = fem.functionspace(msh, ("DG", 0)); kappa = fem.Function(Q)
    if interface:                                                      # from the Gmsh cell tags
        kappa.x.array[ct.find(1)] = K_IN; kappa.x.array[ct.find(2)] = K_OUT
    else:                                                              # mesh ignores the interface: use cell midpoints
        nc = msh.topology.index_map(2).size_local + msh.topology.index_map(2).num_ghosts
        mid = mesh.compute_midpoints(msh, 2, np.arange(nc, dtype=np.int32))
        kappa.x.array[:] = np.where(np.linalg.norm(mid[:, :2], axis=1) < 0.5, K_IN, K_OUT)
    V = fem.functionspace(msh, ("Lagrange", k))
    x = ufl.SpatialCoordinate(msh); r = ufl.sqrt(x[0] ** 2 + x[1] ** 2); phi = ufl.cos(ufl.pi * r / 2)
    u_ex = ufl.conditional(ufl.lt(r, 0.5), phi / K_IN + np.cos(np.pi / 4) * (1 / K_OUT - 1 / K_IN), phi / K_OUT)
    f = ufl.pi / 2 * ufl.sin(ufl.pi * r / 2) / r + (ufl.pi / 2) ** 2 * phi     # = -Δφ, the same in both materials
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    msh.topology.create_connectivity(1, 2)
    bc = fem.dirichletbc(0.0, fem.locate_dofs_topological(V, 1, ft.find(10)), V)
    uh = LinearProblem(kappa * ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx, f * v * ufl.dx, bcs=[bc],
                       petsc_options_prefix=f"disk_{k}_{int(interface)}_{msh.topology.index_map(2).size_global}_",
                       petsc_options={"ksp_type": "preonly", "pc_type": "lu"}).solve()
    return uh, u_ex

comm = MPI.COMM_WORLD
for interface in (True, False):
    E = []
    for h in (0.2, 0.1, 0.05):
        msh, ct, ft = disk(h, interface)
        uh, u_ex = disk_problem(msh, ct, ft, 1, interface)
        dxq = ufl.Measure("dx", domain=msh, metadata={"quadrature_degree": 8})
        E.append(np.sqrt(comm.allreduce(fem.assemble_scalar(fem.form((uh - u_ex) ** 2 * dxq)), op=MPI.SUM)))
    if comm.rank == 0:
        print(f"P1, interface {'meshed    ' if interface else 'not meshed'}: L2 errors {np.array2string(np.array(E), precision=3)}"
              f"   rates {np.round(np.log2(np.array(E[:-1]) / np.array(E[1:])), 2)}")

> **✏️ Your turn (5 min).** Interface meshed, **P2**: compare straight-sided triangles (`order=1`) with curved ones (`order=2`, which calls
> `gmsh.model.mesh.setOrder(2)`), $h=0.2,0.1,0.05$. Which $L^2$ rate do you get in each case, and why does P2 on straight-sided cells not reach 3?

In [ ]:
%%fenicsx -np 2
# TODO: P2 on straight-sided vs curved triangles

## 6 · LLM-assisted coding: live demo  *(lecture Part D)*
Prompt (paste the API reference sheet first):

```text
Context: [API reference sheet]
Task: DOLFINx 0.11 code for a %%fenicsx -np 2 cell. Solve -div(k grad u) = 0 on the unit square, k = 1 for x < 0.5 and
k = 5 for x > 0.5 (DG0 function), u = 1 on x = 0, k du/dn + 2 (u - 0.2) = 0 on x = 1 (Robin), du/dn = 0 on y = 0 and y = 1.
Use facet tags and ds(i). The exact solution depends on x only and is piecewise linear: compute it, and report the L2
error and the heat flux through x = 1 for N = 8, 16, 32 on rank 0 (P1).
```
Paste the result below **unchanged**, run it, then go through the checklist.

In [ ]:
%%fenicsx -np 2
# TODO: paste the %%fenicsx part your LLM produced (unchanged) and run it

**Verification checklist** — checks with a known answer

- [ ] Each tag covers the right side: $\int 1\,ds(i)=1$, and the Dirichlet DOF count is $N+1$ (P1)
- [ ] The Robin term appears in **both** forms: $\alpha\int uv$ in `a`, $\alpha T_\infty\int v$ in `L`
- [ ] The exact solution is piecewise linear and the interface lies on mesh lines, so the P1 error is at round-off level
- [ ] The computed flux equals $(T_0-T_\infty)/(\tfrac{0.5}{k_1}+\tfrac{0.5}{k_2}+\tfrac1H)$
- [ ] The numbers are identical with `-np 1` and `-np 2`, and printed once